# Heat matched-baseline pilot — T4x2

**DRAFT_UNVERIFIED; offline review package.** Not uploaded or run. Kaggle CLI hard timeout: 7,200 s; allow 300 s for startup before the notebook's first cell, whose watchdog is 6,900 s with a 60 s close margin. Tier2 has 15 runs × 500 training GPU-s each, smoke has 3 × 120 GPU-s, and aggregate device cap is 14,400 s with 2,730 s allocated to finite score allowances and 3,810 s held reserve. Two independent processes use the two T4 devices concurrently.

H-FNN is a Torch-only port of the tutorial architecture with the shared Adam-only recipe; it is not DeepXDE execution or a literal reproduction. All scoring is on CUDA. CPU is for artifact I/O only. The protected 512×256 grid is absent.

In [ ]:
# Start session quota timer before payload extraction and training.
import os, sys, time, json, io, base64, zipfile, subprocess, pathlib, shutil
SESSION_START=time.monotonic(); SESSION_CAP=6900.0; SESSION_CLOSE_MARGIN=60.0
DEVICE_CAP=14400.0; DEVICE_SPENT=0.0
WAVE_OVERHEAD_RESERVE=20.0


In [ ]:
PAYLOAD_B64 = "UEsDBBQAAAAIAAAAIQCJbMfoxAkAAOQkAAARAAAAYWZfc2NpbWxfbW9kZWwucHntWltv3LoRfvevYFWgkGyt6nWRothmA5wcnBRBnCBo0r4sDIErUSsiEqWKkr1Omv/eGZKSSEm+1elDD84Ctle8zHwznBtH9jzvI21oyVrWrEraJjlLiWBdQwtCRUreVUVZHaqmul791IiqSElZpayQJKsa8inh7y+JTBrGBBeHyPO8k5OsqUoSx1nXdg2LY8LLumpaICaqlra8EvLkxIy1VZPkeoP62q8VwhqMhOjHs04kSAGxSfLm5OQkKaiU5O+v31zSW9b4QkTvq7QrWLA5IfABQG/4EST6G+2k5FSQhqYctu+pZJLURScJJW1DuaD7gpFP/PIfhKUHNrDSMiGtlGUgFhe8jWNfsiILCRd118oN/G1DUnWt9QQMuIyTqhOtGiFbsl4bVPiRXQ14g2igGIxTQDtq2IFLOJN432UZrPQSJuBReqHRSsGFrGnC/NU6On8REv3bYhs8RPGGp20+0GuZkFXjn0cX6+nOpGJZxhMOCCTIAUoeTMbXm1lZt7e+0UCvl3vR4AE8gZa1HbagyiJRNSUt9FE4EEMi23R7Hq3JH4mvt5+enkcvHqKBkO7ZOxiBkspYwHFj1PdZqS8gq1fOwHjgDQN/EGaSHWt/5fvHqBPyXx1jX/EYYbPRtz7qADCoZ3VQATk9JRcWDHDAG9qkzwQCVtSVvrfnh7Dih9WrfQUW0asDxDwG4dwIAnJG3qAFMtr4A2H8vIkkL7pxF+p0WBAMDvv6U42773Tan7s9T0hVM7HqBAdJS/J6JdUe1z0lueFtTjLl5F8gwsg7HNaxvnB4sj14GHQ8eRhd9OhxOmUHCIP9zJ/GiaK62ZCsqCiOa2cd5nJ+yMfJce5xgYJnNijyctuDcI4Eghu42j9p0bFfmgZc3LM3lZ1sCTsmDPRnFKyJeA4f0CwKQl5qyA8wMIQ4mvE1RFvFJKfXjNSV5C2HLzr4WDxwLa8aA2vrSLYymODLerYD1ro2OImPADtUqMMpkzPwp916s1pfOftBXHehMwu5jxE7Up0HloH335QpAjK9MKGtC9F9GkFnXVH4vhH2jKxD8CPAH4Sz9T3C+czdlFAJE1JL2JezhZIIYoP6O1ls2PT291v2+OHZA4LaL+BJHW3B+HJGfq6Oq5SR1xXYv6IYYlyHMgbKMC00oV1bHaDcAa4Nv6bodTo49jQT2JxC4dOiKiepyI166GK+tfzVVh+DsoUdOlBA/kDsFS/tBevNVRBEbQX5Lm1vazZS/z15q7QEkEEsYASJEdCWcBCk4AzU2fCkLW5hXvJUi17TNDXBfogwkUXxHWM1agHXMpHWFUbkpBLXQBQyBinpFyj9cLbA9ENuWFGALjOIWCn6PsxY1IYYBuAKkKwhnYR1KW+YAkaxgFQlMUmrEg5ghELbuAGPa+MBxdbSeWQn/q2jUDsgAaAZnYiKWz9wg7DFQ1vQV9ZUMi74F+arMwwWl++iKAohK13BvjmfpSOzjUJzuslZw/z5bsegwoGlcWzjEvjBqwSoBXTLsUAXB+avQyeyQACbyFuwrI1TJqoStamygKVBRQ3tsi+pjKWSla8ZIUE36mvw91PEbWQzIQoJRE1ezfHdMCSK3mO5xn2IsOSbShZByVTWMTzdmTW4yMaTilgt74jxo5wjsrtlHDEjrJl+fjAuV2G2Zc2O+hU5D+31ITmPzu+V0iY3P2hFz96xQLA3eRvmqR7VPqSs7cxl7CyAKDitwdXsr6Oij9/99OE1TM2qeRUb44yWvLjdYEAf5dX5k39lTxcZwmJOJW3bxj+GxNOXV28SI44qr+m5YeIYq7IQpi6iczih424Dx71RoQIKcKSs5Kybal+wEkOzlzPaerryG5Yvax8rPd+wCG0O682F4QCKTHm5XVs1AWQnqZoQy3oA46Q3T1HNsgCTqt2GLYG1/lZzWydoFiN8W1mnxEdtrey1p4jzXhy0KJiIE5qLCZr+TObqxU8LFUE/iUjcch1KMWzpbDURzKsU3BzhmDOppC2cWhUs6aKndKYZDiK6dKdyms04NPXkGC+nMZwsXEPcC+j0iPWtTx1qVxds5x5/AaX4LoVKaAdmEjobr66ufoRX5H22s3zyOCpJiQAX4nuBAIndeDSY1LlIGQDRdRbkdia6kjWQUHTWUeNyAk+x2pA7mACPb9+nt7VeZkUP5FZxbir2QHtn5pGW2mHiYu7aRN0wmrSqcIai0SydLjIEnbWKsDPibMmHdNRSkfvOwkDfPUFpUECDpzhqwuihY9DdxA2eCZh8vkZGtK6hIvPVUzA1ZpMf+qgEZwgXR7Pz+flK385CEvdGt+Auxxkmvcvu+X58C0nH7zPPpOW70l00XEdgRX8/gtqd6uqfHcGY8aKgq3eQtm2qYmwf6cTFUwDpNfssVj0mb5bTzPTdTWIT/1Tue1ZfGGyjj6X67kN8HdjR5LvmoLvDdowNHmrUdAIqvxvR0/Um1+whdPcL3GltmPoernP+JajULR92Q2+egvDupV7FCBxGWb7y2t9dQHz/M/5chWSnv8HPlVWqz3qI9xoBSVQjcegdwiq4v/W3Q0lL007ENx06SqV4+ZTYWVy2hL2mFCvCy9ZgljzLIsJpP/HXbCKzOO20h2d20+smmO17tD05O5eM6+PbDx/eX35csKvPELXVtO48U5LzFC4TuqFJkrySDKsqot6lKTPDCFT3bSqiRFi2rbKoY0wKy2YFs481KYWlN50Xf/k/tR2t0a2W5r8wK5i71BeaC6MSyGHjoBq5Z2LtRB3bxIZ6xsKiumCuPvo24ZFec9bE5kVGrMuUqL9pHiApbfulCS2SDttMMQ77nrKGIFgkqxs8htqeUzlvUE4523gBrsEw32dIT5drJr8Vuc8vch9VXP7vK8WhCvw2KWE37k50e3sy//7EivFEdb47XqSxCmu+G69gX8YPuugn/yYfKoHXPfyjrGVw8vH9IKBrMW1jw7jgCcdWrIAIa/494a8q6DYMG7K62tvT5Ataq1wlVVmDIPi6H1DRrmiHrrjGoe4YqF/zyKVGpDSKEH09EUwDuB6ODqz1PZ56oTPQr8MYOpaTxreB2UjHqTdnXZ6h7jUqdFLjFkK+b3O15oAxVJp3MHTLmhlTp9aaWLJGMW+WPRqRu1fn9tk281IyhErI2rAszJBHZ3L0aX3QnYr1M2b9f0ZA6jT6muW2bEhu+vrA0w351qP4XfPdC4zVW29f/NtpX2fxzrQQG6MoMhEQ3+hgx26IDf1rngh/jWdzC8T72AXGa94E3IJPJsp3Ylhe59vPTcf6q5E1pCP17lxHug6jac/YadB07WRm6M50x+NDELvjIsbu+FSQDiRz0kgc75iA4+Q/UEsDBBQAAAAIAAAAIQD79l/hpAoAALgjAAATAAAAaGVhdF9jb21tb25fZXZhbC5webVaUW/bOBJ+96/g6V6knK3GvnZ3z3s6IGia3QB7RdFLFwfkAkGR6JgbiVJJKrGbzX+/GZKSKFm200U3D4lEDoczw5lvhqN4nve2LIqSk59poohMS0EFWZWCZPSB5mVVUK5IwjNSlVLNlEgYZ/yOpCVfMVEkipU8nEzOP55dXMWf3v/67uPlxeW785BcrZkkRZnVOQVWObulIlE035J7Xj5Kwku1Rj7JbVkrQh+SXL4isKta00klSkVTRTPyZr7YLN58R84ujGTkTrAM2K1ACL0zuTSLa+ANTOuCCpYmuaaTKPakoAqGJAENgTdJpGR3HFi//XR+hjqylP5I3n74REDcWsJEyUFIVRIJrJKcfYE1QrFVkioZTjzPm6xEWZA4XtWqFjSOCSuqUqCNQCdtDzmZ2LHfZMmbZ8UKatZWiVqDQZqFH+DVTKhthTax42d82zICzaotCE941fIrRbqeTCY/fbw8jy/PSUS8NRzhDHSaLd58v5kv/jF7mHuT83cXZ59+uYrf/xdIYKIbuIIBoJqc/fLh5zN4Pg3nwA+sS2IUPH4UTFEfpV1qIacg+TYvk2xJMpaqa6nEFKW8CcjsX+R9yelyQuAHV4R6cazoRvl6EH+Qa5iBKtK3nKaE8Qw8LFpMiQSt4nu6ldGVqOmUJHlePsY84dEFeAcNyN+I9z/uTVt2lKdlBgaLvFqtZj/YmaDRAXy1ih8pu1sr6adlzdUSdlNTe+pLY8HQvGkVzMAV5SCKUcUuB+OYOdDR8gI2cFw0MuMrUEZ997rhHVmmLo/r69Mpmc1vbrSl3+gpQcGHeEPRCF4Jivb1IXpo3ojJefhvHU1TiL1SgN7o8suezPuUYGrdcCnjO5FkfrBsrYjRQ1FDvZ3vMA9CQeU6qag/mweuvGaJFVfHIo1lUd7TvSKfdFYHr0FfTessWZ56WuS+NxnRINLerml6b/kTgImqVlKH5494juA1EGvgJFswSAFzCAAl+VxD0KotMXEfYsAiO0lzDSmxEaM9UPPqu+fFVkPqEE+aRFZqT8MhBLtlgWMhk3HykLA8uc2pa13AS0nJR3AYCP93QpTC9zT0ANwI+rlmcNYabhGcjKozjUA/ASRpm5IUzRB6Rjht4OawQlVayaOBxKPe+UL1OuEd/eSWp2tRcsBDf7DcsJUKYBJ0iTTQhRUVq1gHChW+odi0Rs8Zl1UCZj8NISbm+Ov7w/E0uqU6xPDNH2C42UyJ6rgW4P2YSHwcNki10XDDfrOn4QRLuwoMkd77/mbjRg/ydd9hIGNFZMOqDcF+6Pci3VDaYIjIbVnmvtmQSTPqGzZBCDHhByGMFH7wp575XVXHkkIpkGn8SDbG/iMOQGaNg/SA5Knd1rMwYsI81j7vLa2+HeR7VQmhL808zTyN6FbxENM/aB445BYNYl09xBYncJlOKQ4hnnPMcAbzmDNhqwuAGqu6pyFsxyDjSxwLxVSCYeAogYEzbNY9WzA1cuoziA0ed1G/C6t6Dixm0jP5HQUzgyfmD99oA8HhdFWAnVE7M1dmJsmrdQKWx4CBaV0cmJlxBDdz1n5m8ndtRKAxthxgvObcofx/dGGH8JeaQtQtPHW9B2MImSNlGgJnU50RXXHIFvHB5/mG/JP8HctKYAVPQ1j+Ff3GgvL5cFeIT8ihWM6RopZQZibpmtxCNahIThMYUGtBaYPLGZ4u14cOeuNx+HAwO3NhcZ8xAdWPgH2aQgdQRaq4vNevwbdKV39p0xXq/8ez1VV3Ml19bV0cONskJkeq66vX3y5rmZW4LWSTyZ+QnDBUD+UTvvkjGeoo1yN15J40hWwxpfTYd+nKTps/h9NWj0ObunB0mL6GY/0UdrjGtGkN9DDb7ZSaON6Vm2hrboOHbiC2+1IybjNfxciJtUYAT2aQbip/pkGsHarYyckC3zQjyxe92/Ltizfr9jSe9oiVS/86gRIOz+Z6OdWAd2MWqd1FamSRhkiyNIsgXWVwTcZ7ZXfzgO1PgJ3N/zTPYkFzQJMHGueLziqfhXPTsmN14Y+wPOmUB7sE5NVR8sYeSG4vWY40haR9MRp/TLjf28ldBdVCnNx2ZRNWD+YJRrtlgY12lsaHii3r8XrsCxWljHMG1xHjGn1PBU7teY+UXP2N2iXaBHtdsHMrJEXZgbS30axlYvTJ6Uod0mhHFeO6Nqr7Ggk8p+PM8O56lNctoGKWiG2rhYXZxHEuf2izoTLAckiyI2PQOJKJRk7F3baTGzxRu30vMsGFjKyn6LMQg2DVuYsSLRe4X/fxYBGewsAeVHBAQTPYH1+OY/t2s1lvb4ChE+KElIkBh8CEgd7srwQbUJCCCihyoUJIyWJ6+vqHmS5xiaxvJVWkXOm7rmA6iUMlYU7IpFvMspZVUqsSURewHUqTxzXLKbmnVPeTPpy/gzQtWQbFMDEFNGHYddGp/HZr6p3L89B4cHdPS0TC76g/x+yHtu7nqrzkd4cTFVNjrNTXsjG1JSCueRi5oOEkG7+iNYrHMAccd2LDMu9nO7vR/nSniwdMZAAECzi06c4+oV55fXrTL+hGioEDa43RX9BvGtpNW9Uf1JKdBXY21IaLDSVK1KwZ2HA/xnTgYIqHfQybHTFZgm0sBr2IfH7THoKBDqiGYe+1j4WDKUJ14RH7XR3dSj7oco1p1DhbkjHKHc+1kRXqoqYrlfucp7sTDutuVgtoe1nREJcHPJ07ZSoo8EH1qrW5NTj7qYTxnakA3Ec/1BjPjVba7Mu5nVGDmfly0azZHNO/3nwTlevNPi0HN/Sems4c6tnTqspo3KJdhErOWuAHvXZosH7Zi/IuYetuBuODr76FjPcFvqJzYm5CurEZK93g3R+G7RP+7JSN07HpwrV3N24rtf7UuLma6kdnubEF/SqvIR+nHTLvlyYv3KK/aLhi6Af92d1qoJt3y2DsNMD9eqwl1z+sQWtueO++QFCH7PUBU7S9fZ+NdUUMV2yF8JLPmga56ZK0V26zc4t71zplmFZZoBsn+hFS5sCjOlRNqxp+qzKH6sQPbhy2yFBXd60GX1jVd7n+G/54O17oTfcSgS/un7UeOUYAeuxbC1MHFt7uX3h7cOHQh8Zodj1pQDXwy97h7TqdwTDzBQ1OwumiAgqoWnpL4g2/yDobOs1O+4QtIr/5kAju3NzAsUHsO61Dp1kYEAoYTFbdR8cnvnnePHH1PEtrqcrCCwZbwn5PHt/Ypi0H5CceV82rwlecvG7aIzcwoHoDzw5HjemebVT6+s3dsD0SU/DaXTSsjpLJpKigJrtD2/Wr8qZSI039BIzKrip3AnOmC2ldprvmxqYucP6G/eNxCocFhG4syhxpvbZB2nzRNl1KbKC6UrZf3WP71T3WzoE+Ntox30OvO9hI3/MhIJkSoHHVtFACpPZp3ASPgJmNhkj7slzZtNXxt/s12+3TviIebMMy05lFIq/9xB0cW2llbpY9mUB49XRPt88gpgFXhFkYmA6xVuoUIP3g2ezDq1AmD/SL/kIBLimpU2ntF5lXX5wT3ESm9B7gt/7XAd8xvIrUi+js3Rs2inrX8KMLBV1RQTlcS9rG0dE1Oj1HbdvnOL1G1Mj8OU7t3MAj9+UFuuyWttHY4Ms5GVSPRkvLscW9L2bWfyb/B1BLAwQUAAAACAAAACEAGGcVNXsVAACBQQAAGQAAAGhlYXRfY29tcGFyaXNvbl9ydW5uZXIucHm1O2tz20aS3/krJqi6OlAmEVJ2cgkVpkqx5VgVR1bJ8u5d6VRTEDEksQIBLB4SGa3++3b3PPEg5ezeuRKbwPT0dPf09Gsanud9FslyvMjSKoxTEbGiTlNRsGVWsGot2CasFmt4/UGEFQuLTcnilAZ+C1erRLA0q8Rdlt0Hg8G7q9P31/zLxV/Ors7fn5+9C9hVnbIsTXYwp4wjQfMK8RCLR1xIbGCuRcCu1zFgL1nIUvE40Asv6yQZ3+EDy4usyhZZcsLiikWZKHEyWyRhvGFVBggBIKoXAgBjIP/y/OIU2SmDged5g2WRbRjny7qqC8E5izd5VgBPKSAJqzhLy8FAvytWeViUQj8vygf9cx2W6yS+049/K7NU/85K/asI0yjb6KdyZwaqeCMkIXlYIR5NxSU8yoFql8fpSr8/TXeGqrTe5DsWAtf5ICsDkT7ERZYGpagisQzrpPK9t19++Xj6mf/109Vvny9P357xt58u3p//6o2YN3sz+fH72Q/e0NCSFYv1YPDh7OqMzYkAH6QTJyCbYVCIMksehD8MQBAirQbxkpVV4SP0kMQOagCMBcjHbMDgj34KYLNFUfmTkZ0xHEjmwiUvF/Em4ZssEolm8q6Ok0i+kmBrUDa+yDabLOXiITSAyziNK8HLTXYvAPsig40kNtTkweD04+WHU2BnEkwHV2efz999Of3IP789/Yg8TuHl5adPH/kFPKA4Br98+nLx7vTqf+jNd9PjwfnF+fX5qYSYTo7fDD5fn13yt6eX+Mwnkwn+P/h4RdgmYvx6cH3+O4j5w9nb3y4/nV9cf4YRf3o8CYD91xP65zv8Zzg4vfodB59IWF5xt/Rm7MkjwnkcwQO+gw3Yigj3KwnvRIJvP4yvfnmPb2Anwo2oRFFysc3FohI46/X3b354Hkmkd2WewCHuIFbv+aK+ixct5L98/jrcyzTt4JWyr8J0zWGYH0+2r/k6LCJeFeFCtBZ6f3HB/HdC5P/97oxVNcyNYWdDwAB7usBDyXCTh/vJ+fF4CtQ8wzaDwtPR448FTPZJCUmFRwzUpRYzFsWL6gb0b4RH6HbIxj+ziywVUlVJTaViB5v7KC58+VDOr4saNEts47Li2T09Du0UWo1XYlv5uHoQwYksfVpxBAciAhTzY1BMYIPfi51GFyZJ9sjTMJ2/D5NSDNkr5v1vCnyKdJFFcNrnXl0tx3g2JWvlLl3AZkWh3yIcTiHJPMDBIC55+BDGIOMETqqEwD8OCGJag5WI/wAIhT3PsoSHRRHuSr8UIpoB6RUtZIWW5kEaEcytxEtkZXdZ4i+yOq1ozggmbASY9ixVzzBXhBvexGpxWRJFugKFhOMgaf17HZaxtJrBZ1zljMZ9gx9PO6gEMKpkikvMneWGBnUhQJdStUIQFeGjJBmoBdsq5nLFZZKF1fdvhsEir8HKkW1FCZGUU9A89CBzxbI0GiMGezudsCMEIMnhTk7lynewBDC545WZZY0LTOuZ+Xp4Mxuxya1aM67gPPCtmW5MUf/sN+5stITGttBx1Tx4M8POyI4mYlnBCOwMuNN6k/KyChf3vg8v/hBFVjq0D0cOb8Ohg6SIV+t9WEBjvw6J4rsPjREJqaOkywhlqLE809/aSvByHeYQGIAsXBHYHRyODPeNDaIBzVF3xJLp7svxUK6OwVIKBgvUEpdH39giKADDsSndQwpHGXftBufdBnLeN3OJwEKRQodxKdhf0MqcFUVW+EvvS6rxsydE8EyHWq3+1Mb7fMIsuHwTeFJt1WHBGa51wCjHx1+zfptA5xpeKtsQr0SJeq+CI1z1+Lvv/WFDOBQxgGUUEWF2REE4YTosEJYYicarOqtLek2wkhl9gL2flj949rzL1YM6j0JwBQgZkF0Vvraqe2FxPbkK/S2lZZeJYW5QZXe7CpR5LxY5tQWm5Kog12IrfxkbLD0J8tZwXk3D6Tgs3N15x3IP/1VvBpyX4YP4A4OsHGK9EjbFsIf47Ak9OsJV7TMZXkdy+KxFBq+BdrCqFpxIlvrgzmpqmWsRVgLSjxCMtAvuHXQTxjdEJ0ReCdbyCH+8mn776vW3r954Dn7UL4hOqgZ6DDjH02gcJvk6HEPweKL8wwmGiWNtSU4wJByfvz2BUHGci2KsjdoJ0Tcmwz1+mOr19G6Dl4pe2uwDvvcxrtb4FkiKCIMOKfJ4cQ8eUUUVkBngpkazhvDRFOKhIAOrtgyhmmcKxhTHI5BQvpO60ji7vjWo2oYak+lYSGUSSVVwHfLOQAT5LlrYwzferXOeBORwBYCiQqD/g7RBgTrqo2dgJmIRfzO3XtEy3mMyL8lACjKBZvqzTSMp38R0UQPhsDaToC91mGj6WsqrqXKBgKwGVwdI8y4hzIUTCosCozBBbhuhaVEXV6U0lmXLfuMMpWsR5NcLMC1Fhrmc39ItDIZNKImI94aTDAsAdlChpUgKBoG/aZsnyPYxv1VcnYVIMGSRYSWrCmxTg5MAsYJBAlElOwbmDZS2jFdYd3j75d0p+/XyywmQxeCEgj5BGgyhAHoMza4b2YqKS6L8iTJqqKhzF2ZlYDgOakBg3aveeDqNJZcBx0kUfkeFGjwtPSpRLMJNHgLRIPy/1zFYTxamlo/rNydwbMAsyGP3TWGUCLYEbEYVU3jST6UF0bSqDXYiOwmKARNGHPb9Q1zGsHPc3SkZ/fl799GNxGBP4gczHbOZLUyfOBBVVoF+b8QmgzCO3J3Cb8kOXBgX+yLMw7s4iasdzElA3f1+CVg4kECDPLB4C6rT8AX8usek0sluIvYIEKBeYN/BAWQpaZfYLpK6BLaSnd0hJShUNk/HkPLshHku0giEA/6yyB4b1hqe/++zylQ8UskFgx9QRsJAYKU8gPQCzgBsFfzDS2CUzedsYt0CQYD0U98LwQwDQsz0515fdkkeglImq+YUiGCmsygfgnfA3l/phS/hRmwZiyRCTSvntGsgBicSQiOiWGjGrBKtTJjBuUZ4tgZ9gyhmxKkzX9AEXccjbUYz7O8Li1QoYGBMcNM/IDVuE6ZopnvHSBsdAA461QGqSzwlQPoGbFRZxQuAWmXAzBoifLu3EvgOUhnQqRJRp2nQmAdSR/D90HewhetNWNwDJDn5PaBIc7Wpk0CGBdXy9fELMwD5QVhpXamqyClC8JtpM2wXeLeyZNeqkPP+4kKBAOrfYfsSoYyp53nXFB1RJVhWSpdUCtZFoP8s2c3x6Hii/pveMiwpgWZVj1lxH2Dx1tQgOMdAg3PYlWTZUgf8U9Y56lpg4By1i6NqjbbXhzwe1jH/Ty0MYg2ScAcu2Rhpw9BHPAANNb8xEB/h1IWFj2ER2AoMimT05LxAX/NHnPuSjpvZeArhl3qYzm6Htwa1pYdQIDmUPVnimqfNUIE8B9vwIRYFT7NiA2rs04TgURBNB6ZRgq3B7+KwHFqpAxmPYRGR0DFAzAowLODYy5lCci1SyO1oP9wXlkwqkqFMnckvckkyarJq8MhlUFEkybIMVzrGqQgfMZh00AE2DWagtk2aqKgym9q9qLrj09mxHVcOWh2bOFWnII/ZEdtioabCH/C/Pw0mbIzvjpA0bfAgFuWLtVjc5xm6UsJLNdZZW/1Gxsson0TPR/KfsNjM0LjLJ2Mw1WMlcucR4xZRUnFCLGaMzjT7Bx2kkRvu2tGRyUK5TGTVWv++B1RyAynYk/XU2HEP/V+NoYbXvmHyRk1IEAKAwd+t907NGqvwNwBxe2Pf3rbAKT+Z2byiPQzS1MPwsz1spbvKa64CEoBvSL1NuBS4M4EDaAxWHQM92gLfbkp7RTdJmrm71CEc8HGMYbCSL5PCis4p+iWI131VDnUKWnIczyWJK7A4AFgVtZ7tOs5PW0bQzp2udZKsLP09Cs7oDHydUcnqKq/xcBIuX86UOrUqwigGtTNWIgRHgy8D/MvqmUTh1ioQh1OKAGgugUpVNsa6Jk/ie+HL925hAQImEA1MytdS310bAflMZ2h4o8q3NRohTXXDAhGH7oixPfV2+xJ/9fZf5K3e/tt8NbiAHCmOQHnR9wJHYyav6Y6QiSH7ljVv6ty8R4VsIkx9jeTo6NjUVepU3f/BifbDYlXOzN1tcIFBax4uSK32VjJHTGXKvcF9X+LcH6Li4oGNFGU2YzZIJaoexmqziWdLBjgLaMegXt6wGdnKG9J5I8CS8wSEal045xJV1rMgA3jyXJsnl2oYvufhwMUhj3iV+ZLeufyn/+JEGXx1TycTSl3mqTe+GcGrFSBJ2hTz1hoUe9UHMCASO1Gn16hf0dDIrL3mN/MOg33Xh7cvpPZPev6zQ6Vc4am15DPoReSUZXSlR4KbdJ9OnD2jJZe2VFa0bTHttle+I9bYAxdlUzDcSTnCKNyYBbMc2AtO4ZXflfSIJcX845VGazzGnuqWDVBKLmuMlMPfELG3MOlGHvQ1vMTiANp+eEsiBSMDR9xTQwHkmZ4JSpjOZaWPh0cPeeA4hqUAfieWeMnvOkZ4LRFA/iHdSTOEAST4T5tulEudJ8KX3pTCQKmU9BMVksi1Uziqh3Ipzo2sphfrUogUgALIOZZSNyjRNXGzSMIca+tkTGhhJ1NpoXSs3ibc+tQ60IMcrKda3FmHosioFhCQhCshgzNGv7mN/RjKiqv7cVdifdTJaAP4Myy4Sb9Fzf5Dii3JVlw8iGJH9QmsXTgw80a1kNC3Ki2NwbYmjTqjTx5h92ZylZEJyuyiWKMGfrHaZNiGd4qdfaGWZHt0IKZ6blLTTNU24EiA7pTUyVG+JvMgQAuoipCdE8bCNNK78PPcTph1hNGbQrT/kAnoCpL2wp7SpWfxjJ8QmVl3+FwGeeXtQ7GZa/vZD0HXRsZJ7oGBbZvT3vUONwLouSFsD0UmWp6rPe0FczZ2vjdwxj/DzpvOjgVSqR2Rtc+1pEQe28c11v3oePwkt0AbvcYdsbUgPyt76mhuUxesDW2ZS7AMRZyuOJpWrzHlDmK7e3vmYZzKAORYfAx0qoynwImTsOEfPFF43duM53X43iwPmjuGuJTtWz7OGO4jHVZTXV50bF8iGYGosEXliWGTFRSo845k/QpMkZ2tPQisLN0C4WsmQ5TluNe/xjQaq0tOyxsxaXcM1oYg7FH+f9xFMs3ApV5FqxQZE4X5mxdQOx7qTzrllsvl5Ke40uCOK9lrtlqmyg0iHOsEWhImTYvUb4X2WZ4ea9O0MO19GrU0gID2sdu6Ad9nYlTcJu82WkKSwT44Lo5XG5oKam6cmLFaXok4ZDoAOsOghklnX/ERcpiyTjDgc7sq9UlWwWczYcE/TteduzESJQkjQBA8DM4qdv5emuX5cyfdeP3A1l/fdkxNc77iTKa5kjrv9mXLI+G9nmSLduNw7Ge0WFRFvEDMnT5VLWQtwAPi7tv+F6JDl0aLCF7GEYkSMFDzInYTlH5jFy2Q3ETsA4YMgxoeO7dJX72nFuuf2tEGZZBixdgHyLFdPNhEQJrTjNksc72DODTJ8o0AK0F3DrQBzHQWj0Ha4+Pv/ms7Pf5x/DDFxnNBveWyQ+r0PWuaV++76fEW4kC2KmK8ZpQ38ig+gMZAF9FHAZMtrtTA3r7waCFU9x95fQdZFDa597fDotGmLC4GpwzGaRFjdxIu7rU5ll21YisWNe0xkBVCigY6AUh1tYI2FhtPG7PbG2tHlQmCdH4TYmDfuIUuwiWZvJdLs6Ys27bMpju4nb/L905ZtqeC21PNcOCdEq6tzLi9i3XK7+OUupjJZnhNY0nnnnlVLIpjrzXPMC1/OKNms3iJ3y7UCVVvPczBMd+eivFrhlWHhLZfdXs8deKB1fNY/lR2YYwHnYGLpd6Ij+Nf3v/6uSFf9M6yG02S5fo0L7lbrkpneNKeCaYCL+AthJWa8vjDDjqwQRy1q4WPnLLj1Pm+Yrgqabc4d9dpRxdfM0e3jezN7rSfdea8ZJFme8xbtzXhQFZp/Purl7EZJjDUyiClLtdZVjmyM/bfoh13gsqOWLCrh9sSECa1zapW41xCBGvooEC8XIQJMLgpkR8T4TpzvuISwsDkpGbqBoLaCig9l/2X7u2DKc3gzJ7rBonRtAyabxJe7hRUjRDdVkFXbnDsuJvmocupBH4HoRpZOzmgK3bTqKMqzM0dWYDvwso6uimkG7+bALxx6ufx0Xb4qjraHvnT8XZ4hB3s2xF+GOHYIVUMx5l+zSvsWTyq+XY7/HYaTE/cr6aweD4uQSAFfXSl5vWeAMQmYyDpu6gn64B5lJ/odJyqmodusX+dRkTD43SR1BHlH0UVLyG45nGmzl0D0m0nAk0vMrKvnp4FkT760PgPGeYgB+7y6nuiyEYHXMVogIQKhz0sKghX43SgwFVYwDEs4DIcAEzUSdHQApFvI0hn47sC29/rshdMfkEDy62ziIIc5OxyR4FEb9NEM1SgXiDHqaDLkXuIHb5hEUM4pyKIE4ogQqaDBmzNgwO3DAEFqA2YhZyiDRszeL1XFVJA9tJY9n3uyQ+w0GcSAxlPDA/Bo58FYYB/qsLeeaqiot7p3nWgGtiXrbb7roNad9ZOJzYRQBaFgvLWlU6jC7cN2t+OSwDN5qi8QO/qfD305MYqZoJpku2UIrE51gFrNsjSp1PVWoE1iBw+D80HQCgQjqOyMN2Vkb7Oh5eFrONLgNNiVePJuaQRPxJgT+McdWTeiQCZPHBj3XAtPyHN4wS0T35kqi9KCFkQRhHSRPh9bzxWmzlGBmD3w4VcBsvCaDprcXg6yW/E6GaFqt/q1iRqfNTVvzKsOAaR6enYbPHyBNyPr56BIfGILdYZeIhyLi8mMLAdHpz1Z2iCWGCsYwY1R7ZyMNXXNVffJR6SIcR/Ywy+XDnq6frLyIMYkmw1pmuBXhTTyeH1nXIP3cW0GElRhefekWcxUjDR/iTzsFClfzugYLrHWk53D486TxsIlNpf6iEAXqa50PheW1LXVDUKvo1Di9nkRafo320htzqLM7SmS5PVPEmBU1zomMt2wbpBMdp+RQ6ll9pat98Z49Qlvkv4tW70Nr3UdDZGjJR9RCmwc76C1tW5W274Sd07NQuf+PYgAehSMb2SFVKRY55Vyl71O/witYyxLbq9MBBnG3XppibdGQeGgWqB3U/DTj/5e2DijObpe2f6Rl72ssAUcOcZpNqxrDGITV7tZipDBKBn/TmCXuhrP16V34HYz1CaG9VxdSpgZfPO1wQ0XBXOl5ymjNhtxpCNF6bDYnjACd6L3Uyhwm5wH55lHgA/5Mcn0mJ6yqy7ifjoxQxsdCCWh3Si9cGuMheUyya7wxe2TtswbRboHCxCliHGZlVMYzinmIlztBOcqwK7NBqDfwJQSwMEFAAAAAgAAAAhAHEE8sNoBQAAkA0AABIAAABoZWF0X2ZpbmFsX2V2YWwucHmVV02P2zYQvftXMOpFRm1lEyRBsYAKLJpdNIcugmCTSxoQtDSyWUukSlK7dg3/9z6Skqz9aJr1wRbFmcf5eDNDJ0nyUVu3dEZIJdWa/fb5/QUrtKqkaYSTWjFbaENmwaS61VsqmVb1nonKkWGirtmoSaq0WZIks8rohnFeda4zxDmTTauNY0Ip7QKknc2Gd2bdCmNpWP9ltRqenWwoYrXCbWq5GoA+Yhk33L71J/fvL9R+BHbaFJtZlBIVt4Vsat7okupBetXJuoyvotiGhOOFbmCQhBncdErBx178pgOkFPXV9fV98QaidCtG3BAuHs7v0Wezkip2Z6Qj7h1MvT/nwY0FfNvXWpTnrJSF+2odAg0/vs3Z8ld2rRWdzxg+XiODYaRc1mxLadK4sPmN6WjBaCet43oblvOTSjzU0c6l/uSs7JrWpv2ZPqUlQPLXC2ZhOt/SPgLO2c8s+VMlAFaFLhHjPOlctfwlmffeBM94pU0qTHPOYHewOLidKZX9ocuu7o2XFfLcsDxniVlVSXzpP4bAEDXNRJr4sOLcQyLL5Dwo8EruqEyO84dgK9vWUtHzAHslXnQrWTwBWin1GHCS/DRqgPSW2BdRd3RpDMJQJZ/VVuk7FZAO+HphjtkpXiiS9FFSwXzD8rEKsguz7hpk5GPYSUuyhZGtL5n8iTJdhkr8Hf5F0iXzCWwmypKLHi9NlstiQ8W21VL5aKBwKI8ENPR3Jw2V96jzJAJ8gmqx0bIgm6chmYtTGhYxePPnQFpCZntzYNlzVHX3/45A3iK+PUL48Rg2HdOOltSTtuhKkUnLxa2QtVjVhHRpM90s6Raeo+Y7WDBnL3L2asKUQIhP2ELbipRILkWxYZVUol6GBLE7bbbIeNNZpIwIZSsKhxyCEkxYK9cK/fXmTdZncnK2Jcfj+elZHxajW+/cRGg9CnG/S8ZJsoM8nE3cmyR4LNV/qCnR+AOyWt+Bf/Pvulcll7uWChdMXrA1cA/fRY3l4NF+YjcbaVlkN8NTgUkCnBUGCwKDRAlH3lJaab1lBWHOxInjNhQnDp7R9JzNRkBib1+93r1++46tjSwDKkaNM10wUSorS+jGc/1Ew0AKUopusTAkwvlDhUXcU9GMkfadM/Ukyk6bC9R3y2tdiFirRdv1KYwjJ7/XL6GKUprsB0xuMRyJ+zGQnpC/JqfXybcnMDOn0xhmHIu4n5+hKspQFtHeCuDu3Zt55odU+phYe1VsjFbyH+o3caDxEcvDBM5Aoypy3jMiGkDOyMKT79GwS0fGhOViXAa3UbOnN2qXI2GTNQbR23en9UOvxg2fXi7LPLT25XhX0Wa/7BmwvH3Vy/+Iv8EHSwAqvU+N2KVn2dniKffZcghPPwYozPycHUbrQsI662fN+08XVzf88/WXy08frj5cvp/4kPSnOrEmL9re6+737l++y081fRs+ZwONppC+mfY7/nmyNWn+YVg/JPB8IttqXXO7EYgjhE8yvqLTe7tTrT4p3pcfSMuoAvlDonb48VxgSTAQoseJXCQC3n+3u0wUPNO7EDy+brshtZwsMopi8iGY5nwa2zoUMZXPVfTiDtdbXDKpgct8tXfkSYDApaFVZ9PtqbV9NUG2f3rAEhCCdxYmoXvwgSIcz5ZqtF94CdUrUVuaKI5yuL4Rj1dTQMASF1o274P/UO++AF8TGm2Ih2chl4r7Bso9W0+mxHYKsHAXDVjH8D258g7lz16CjBN2gyfWZV4kWfTVNP9h3T5eg/qhCtR7edgeYcwtQ7zYdoEHjLtBFKANxv9xmKE+PZOb8eFhdT1RVc/m13Hub4GYvzxwFf+I/EWTc38n5Ly/bcYL4uxfUEsDBBQAAAAIAAAAIQAH3ZJJWQMAAKIFAAAVAAAAcGF5bG9hZC1tYW5pZmVzdC5qc29uZVTbbtw2EH0P4H8Y7JNd7Nokda+fnNhujaILw5c+tAiEETncJSKJKim5NoL8e0e7trNBnngBZ845c2b49egDwAJDFxe/wj/zgY+/r+4+Xi+W76eP9weH6/Uajh980FsYfBjBWxin0QeHLSDfupH0OAU6C6TdQCeLOfLzLn7RTGZDIyN9fU2HWlNLATmebxfrJ2ccPlBs8SFllPRZnbxD42YTaIMj1YaenKY6kva9ibXGgYNlmgrx9jZSjM739b+TH3F+UG/9FGaNan7xbU9H+y0rOKAzoAtk6tFRUN/rAVAJufy+Vwf75GCfHuyz/fbzO6HOf6EfU4rXJweE6AnbCUdmPleDZVLrh476EVRWPEtVwafHywuI2geCgQI8zEwhTP05oB35AtsWxoCud/1mCZEG5OLSLmzl+/YFuGTWhW4HAgPGCLxuqJ9NIAOZVM8qy2ETnNlXfmFdS/HQM1tH7bq27ryh9nR4mblWSZqXBao806pRGknopMi0TkSSZ6RMpkg00iRkrLWFNlVRmEY0WUWpKYq8eHd5SzjW2ncduzeX4zV/hkrJ3KRYiKTMK13lVY620pjoJinSpixFaQQ1uZWNzk2ZEWphVSLzXKokaX7Kz4VxkTG4dqz9DUVwcFaqMiVV6ESiTKnKSqMtyyikLbKCjGjSlEUVWVVRUVRWJkiMISzhjyjW9dgeirCSSqVSUeimQSosUSIqlVSixDKxVMiZeI6VznXeJEkpbSMFNpKxTCbQLg5aZVf8OvY4xK2fe/hnX9jrwbGnTxQa9rsDG3wHMWgezUjzrNYDT2jrejrbRZ69O7rHGLxv58z7aY86YNe0nPDeN7696jcceA6u575zPkAqqnwJN59ACpUuofFTb7jIFOemAvZjew4RO4L17d+AOnjuvfnj2TVyJHrrN3aEyc5TPlBvqNdu136LP/gDaGk1BHJ9HLnRmcmeGfYG1lN3+wJzi59D74GFAXPqafzPhy+vmTlqnHa5Lu8urh/qx/VfV3c31zdXl68P3iZnZ1brccxZyBaDAXpGPc6TpSkuwU5tu+KaMvaf91fHx1M9rsSp/GWqn59PzuSpPGGpHMcMLwx2+8nb/4eAI2RCwG+3j6t4xvpXs/QlSHVwB/vv4ujDt6MP/wNQSwECFAMUAAAACAAAACEAiWzH6MQJAADkJAAAEQAAAAAAAAAAAAAApIEAAAAAYWZfc2NpbWxfbW9kZWwucHlQSwECFAMUAAAACAAAACEA+/Zf4aQKAAC4IwAAEwAAAAAAAAAAAAAApIHzCQAAaGVhdF9jb21tb25fZXZhbC5weVBLAQIUAxQAAAAIAAAAIQAYZxU1exUAAIFBAAAZAAAAAAAAAAAAAACkgcgUAABoZWF0X2NvbXBhcmlzb25fcnVubmVyLnB5UEsBAhQDFAAAAAgAAAAhAHEE8sNoBQAAkA0AABIAAAAAAAAAAAAAAKSBeioAAGhlYXRfZmluYWxfZXZhbC5weVBLAQIUAxQAAAAIAAAAIQAH3ZJJWQMAAKIFAAAVAAAAAAAAAAAAAACkgRIwAABwYXlsb2FkLW1hbmlmZXN0Lmpzb25QSwUGAAAAAAUABQBKAQAAnjMAAAAA"
WORK=pathlib.Path('/kaggle/working/heat-tier2'); PAYLOAD_DIR=WORK/'payload'; PAYLOAD_DIR.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD_B64))) as z: z.extractall(PAYLOAD_DIR)
RUNNER=PAYLOAD_DIR/'heat_comparison_runner.py'


In [ ]:
import numpy as np, torch
if not torch.cuda.is_available() or torch.cuda.device_count()!=2: raise RuntimeError('Expected two visible T4 GPUs.')
GPU_NAMES=[torch.cuda.get_device_name(i) for i in range(2)]
if any('t4' not in name.lower() for name in GPU_NAMES): raise RuntimeError(f'Expected two T4s; got {GPU_NAMES}')
print({'torch':torch.__version__,'numpy':np.__version__,'gpus':GPU_NAMES})


## Fixed protocol

Per-seed pool: scrambled `torch.quasirandom.SobolEngine`, stream seeds `10*seed+1/+3/+4`; 4,096 interior, 1,024 IC, 512 per spatial boundary. The same saved pool is read by all three arms. All use float64, `u=sin(pi*x)+t*x*(1-x)*raw(x,t)`, and full-batch `mean(((u_t-0.1*u_xx)/1.1)^2)`. All three arms use Adam `1e-3` only and share a 500-device-second Tier2 cap; a high safety step cap prevents a 20k-update cutoff from shortening any arm. Checkpoints are saved at 120/300/500 seconds and the final checkpoint is scored. CUDA development-score waves are bounded at 60 seconds per two-run wave; smoke finite-check waves at 10 seconds; final CUDA 512×256 confirmation waves at 120 seconds. The plan allows 300 seconds for setup/pools and 40 seconds for artifact flush.

In [ ]:
POOL_DIR=WORK/'pools'; POOL_DIR.mkdir(parents=True,exist_ok=True)
for seed in [900,901,902,903,904,905]:
    subprocess.run([sys.executable,str(RUNNER),'--prepare-pool','--seed',str(seed),'--pool-out',str(POOL_DIR/f'heat-seed-{seed}.npz')],check=True)
if time.monotonic()-SESSION_START>300.0: raise RuntimeError('Setup and pool generation exceeded its 300-second allowance.')


In [ ]:
RUNS=WORK/'runs'; RUNS.mkdir(parents=True,exist_ok=True); LEDGER_PATH=WORK/'campaign-ledger.json'
ledger={'status':'running','quota_cap_seconds':SESSION_CAP,'aggregate_device_cap_seconds':DEVICE_CAP,'accelerator':'NvidiaTeslaT4 (T4x2)','runs':[]}
def session_elapsed(): return time.monotonic()-SESSION_START
def save_ledger(status=None):
    if status: ledger['status']=status
    ledger['session_wall_seconds']=session_elapsed(); ledger['kaggle_quota_hours_estimate']=session_elapsed()/3600
    ledger['aggregate_device_seconds_estimate']=DEVICE_SPENT; ledger['aggregate_gpu_hours_estimate']=DEVICE_SPENT/3600
    LEDGER_PATH.write_text(json.dumps(ledger,indent=2))
def run_wave(tasks,kind,future_reserve_seconds=0.0):
    global DEVICE_SPENT
    if not 1<=len(tasks)<=2: raise ValueError('One or two runs per wave only.')
    requested=sum(x[2] for x in tasks); max_wall=max(x[2] for x in tasks)
    eval_allowance=60.0 if kind=='tier2' else 10.0
    wave_start=time.monotonic(); wave_wall_cap=max_wall+eval_allowance+WAVE_OVERHEAD_RESERVE
    if DEVICE_SPENT+requested>DEVICE_CAP: raise RuntimeError('Device-second cap guard.')
    if session_elapsed()+wave_wall_cap+future_reserve_seconds>SESSION_CAP-SESSION_CLOSE_MARGIN: raise RuntimeError('Session quota guard.')
    procs=[]
    for physical_gpu,(seed,arm,cap) in enumerate(tasks):
        out=RUNS/kind/f'seed-{seed}'/arm; run_parent=out.parent; run_parent.mkdir(parents=True,exist_ok=True)
        if out.exists(): raise FileExistsError(f'Run output directory already exists; refusing reuse: {out}')
        cmd=[sys.executable,str(RUNNER),'--arm',arm,'--seed',str(seed),'--pool-file',str(POOL_DIR/f'heat-seed-{seed}.npz'),'--out',str(out),'--gpu-seconds',str(cap)]
        cmd += ['--smoke','--checkpoint-times','120'] if kind=='smoke' else ['--step-cap','1000000','--checkpoint-times','120','300','500']
        env=os.environ.copy(); env['CUDA_VISIBLE_DEVICES']=str(physical_gpu)
        stdout_path=run_parent/f'{arm}.stdout.log'; stderr_path=run_parent/f'{arm}.stderr.log'
        stdout=stdout_path.open('w'); stderr=stderr_path.open('w')
        p=subprocess.Popen(cmd,env=env,stdout=stdout,stderr=stderr); procs.append((seed,arm,cap,out,p,stdout,stderr))
    while any(item[4].poll() is None for item in procs):
        if session_elapsed()>=SESSION_CAP-SESSION_CLOSE_MARGIN or time.monotonic()-wave_start>wave_wall_cap:
            for item in procs:
                if item[4].poll() is None: item[4].terminate()
            save_ledger('stopped_at_session_or_wave_guard'); raise RuntimeError('Stopped at session or finite wave-time guard.')
        time.sleep(1)
    for seed,arm,cap,out,p,stdout,stderr in procs:
        rc=p.wait(); stdout.close(); stderr.close()
        if out.is_dir():
            shutil.move(str(out.parent/f'{arm}.stdout.log'),str(out/'stdout.log')); shutil.move(str(out.parent/f'{arm}.stderr.log'),str(out/'stderr.log'))
        path=out/'cost.json'
        if not path.exists(): ledger['runs'].append({'kind':kind,'seed':seed,'arm':arm,'status':'cost_unknown','return_code':rc}); save_ledger('stopped_cost_unknown'); raise RuntimeError('Missing cost artifact; stop for accounting.')
        cost=json.loads(path.read_text()); actual=float(cost.get('allocated_gpu_seconds_estimate',0)); DEVICE_SPENT+=actual
        ledger['runs'].append({'kind':kind,'seed':seed,'arm':arm,'return_code':rc,'run_status':cost.get('run_status'),'training_device_seconds':cost.get('training_gpu_seconds_estimate'),'evaluation_device_seconds':cost.get('evaluation_gpu_seconds_estimate'),'total_device_seconds':actual,'pool_sha256':cost.get('pool_sha256'),'cost_path':str(path)})
        if kind=='smoke':
            check=json.loads((out/'smoke_check.json').read_text())
            if not check['finite_output_check'] or check['quality_score_computed']: raise RuntimeError('Smoke must be finite-only; no quality score allowed.')
        if rc: print('Recorded failure, no replacement seed:',arm,seed)
        if DEVICE_SPENT>DEVICE_CAP: save_ledger('device_cap_exceeded'); raise RuntimeError('Aggregate device-second cap exceeded.')
        save_ledger()
smokes=[(900,'rbf',120.0),(900,'bspline',120.0),(900,'fnn',120.0)]
run_wave(smokes[:2],'smoke',future_reserve_seconds=5950.0)
run_wave(smokes[2:],'smoke',future_reserve_seconds=5800.0)
order={901:['rbf','bspline','fnn'],902:['bspline','fnn','rbf'],903:['fnn','rbf','bspline'],904:['rbf','fnn','bspline'],905:['bspline','rbf','fnn']}
tier=[(seed,arm,500.0) for seed,arms in order.items() for arm in arms]
for i in range(0,len(tier),2):
    remaining_tier_waves=(len(tier[i+2:])+1)//2
    reserve=remaining_tier_waves*(500.0+60.0+WAVE_OVERHEAD_RESERVE)+8*(120.0+WAVE_OVERHEAD_RESERVE)+40.0
    run_wave(tier[i:i+2],'tier2',future_reserve_seconds=reserve)
save_ledger('tier2_stage_finished' if len([r for r in ledger['runs'] if r.get('kind')=='tier2'])==15 else 'incomplete')
print(json.dumps({'status':ledger['status'],'quota_hours':ledger['kaggle_quota_hours_estimate'],'aggregate_gpu_hours':DEVICE_SPENT/3600,'runs':len(ledger['runs'])},indent=2))


In [ ]:
# Confirmation scoring is deliberately a separate stage after every training run ends.
FINAL=WORK/'confirmation'; FINAL.mkdir(parents=True,exist_ok=True)
def score_wave(tasks):
    global DEVICE_SPENT
    if session_elapsed()+120.0+WAVE_OVERHEAD_RESERVE>SESSION_CAP-SESSION_CLOSE_MARGIN: raise RuntimeError('Session quota guard before confirmation scoring.')
    wave_start=time.monotonic()
    procs=[]
    for gpu,(seed,arm) in enumerate(tasks):
        checkpoint=RUNS/'tier2'/f'seed-{seed}'/arm/'checkpoint-final.pt'
        if not checkpoint.exists():
            ledger['runs'].append({'kind':'confirmation','seed':seed,'arm':arm,'status':'missing_final_checkpoint'})
            save_ledger(); continue
        out=FINAL/f'seed-{seed}'/arm; out.mkdir(parents=True,exist_ok=True)
        cmd=[sys.executable,str(PAYLOAD_DIR/'heat_final_eval.py'),'--checkpoint',str(checkpoint),'--arm',arm,'--seed',str(seed),'--out',str(out)]
        env=os.environ.copy(); env['CUDA_VISIBLE_DEVICES']=str(gpu)
        stdout=(out/'stdout.log').open('w'); stderr=(out/'stderr.log').open('w')
        p=subprocess.Popen(cmd,env=env,stdout=stdout,stderr=stderr); procs.append((seed,arm,out,p,stdout,stderr))
    while any(item[3].poll() is None for item in procs):
        if session_elapsed()>=SESSION_CAP-SESSION_CLOSE_MARGIN or time.monotonic()-wave_start>120.0+WAVE_OVERHEAD_RESERVE:
            for item in procs:
                if item[3].poll() is None: item[3].terminate()
            save_ledger('stopped_confirmation_time_guard'); raise RuntimeError('Stopped confirmation scorer at its time/quota guard.')
        time.sleep(1.0)
    for seed,arm,out,p,stdout,stderr in procs:
        rc=p.wait(); stdout.close(); stderr.close(); cost=json.loads((out/'confirmation-cost.json').read_text())
        actual=float(cost['evaluation_gpu_seconds_estimate']); DEVICE_SPENT+=actual
        ledger['runs'].append({'kind':'confirmation','seed':seed,'arm':arm,'return_code':rc,'run_status':'scored' if rc==0 else 'scorer_failed','evaluation_device_seconds':actual,'grid_id':cost['grid_id'],'metrics_path':str(out/'confirmation-metrics.json'),'cost_path':str(out/'confirmation-cost.json')})
        if DEVICE_SPENT>DEVICE_CAP: save_ledger('device_cap_exceeded'); raise RuntimeError('Confirmation scoring exceeded aggregate device-second cap.')
        save_ledger()
confirm=[(seed,arm) for seed in [901,902,903,904,905] for arm in ['rbf','bspline','fnn']]
confirm_wall_budget=((len(confirm)+1)//2)*(120.0+WAVE_OVERHEAD_RESERVE)
if session_elapsed()+confirm_wall_budget+40.0>SESSION_CAP-SESSION_CLOSE_MARGIN: save_ledger('incomplete_confirmation_budget'); raise RuntimeError('Not enough guarded session time for all confirmation waves and artifact flush.')
for i in range(0,len(confirm),2): score_wave(confirm[i:i+2])
if session_elapsed()+40.0>SESSION_CAP-SESSION_CLOSE_MARGIN: save_ledger('incomplete_artifact_flush_budget'); raise RuntimeError('Not enough guarded session time for artifact flush.')
def finite_flat_metrics(path):
    try: values=json.loads(pathlib.Path(path).read_text())
    except Exception: return False
    return isinstance(values,dict) and bool(values) and all(isinstance(v,(int,float)) and not isinstance(v,bool) and np.isfinite(v) for v in values.values())
def row_for(kind,seed,arm):
    rows=[r for r in ledger['runs'] if r.get('kind')==kind and r.get('seed')==seed and r.get('arm')==arm]
    return rows[0] if len(rows)==1 else None
def smoke_campaign_valid():
    expected={(900,a) for a in ('rbf','bspline','fnn')}
    rows=[r for r in ledger['runs'] if r.get('kind')=='smoke']
    if len(rows)!=3 or {(r.get('seed'),r.get('arm')) for r in rows}!=expected: return False
    for seed,arm in expected:
        row=row_for('smoke',seed,arm)
        if row is None or row.get('return_code')!=0 or row.get('run_status')!='gpu_seconds_cap_during_adam' or row.get('training_device_seconds',0)<120.0: return False
        run_dir=pathlib.Path(row.get('cost_path','')).parent
        try: check=json.loads((run_dir/'smoke_check.json').read_text())
        except Exception: return False
        if not check.get('finite_output_check') or check.get('quality_score_computed') or check.get('points_checked',0)<=0: return False
    return True
def tier2_campaign_valid():
    expected={(s,a) for s in (901,902,903,904,905) for a in ('rbf','bspline','fnn')}
    rows=[r for r in ledger['runs'] if r.get('kind')=='tier2']
    if len(rows)!=15 or {(r.get('seed'),r.get('arm')) for r in rows}!=expected: return False
    for seed,arm in expected:
        row=row_for('tier2',seed,arm)
        if row is None or row.get('return_code')!=0 or row.get('run_status')!='gpu_seconds_cap_during_adam' or row.get('training_device_seconds',0)<500.0: return False
        run_dir=pathlib.Path(row.get('cost_path','')).parent
        if not (run_dir/'checkpoint-final.pt').is_file() or not (run_dir/'checkpoint-500s.pt').is_file(): return False
        try: cost=json.loads((run_dir/'cost.json').read_text()); dev=json.loads((run_dir/'validation.json').read_text())
        except Exception: return False
        if cost.get('lbfgs_updates')!=0 or 500.0 not in cost.get('time_checkpoints_written',[]): return False
        dev_metrics=dev.get('metrics')
        if dev.get('grid_id')!='heat-dev-257x129-v1' or not isinstance(dev_metrics,dict) or not dev_metrics or not all(isinstance(v,(int,float)) and not isinstance(v,bool) and np.isfinite(v) for v in dev_metrics.values()): return False
        if not finite_flat_metrics(run_dir/'metrics.json'): return False
    return True
def confirmation_campaign_valid():
    expected={(s,a) for s in (901,902,903,904,905) for a in ('rbf','bspline','fnn')}
    rows=[r for r in ledger['runs'] if r.get('kind')=='confirmation']
    if len(rows)!=15 or {(r.get('seed'),r.get('arm')) for r in rows}!=expected: return False
    for seed,arm in expected:
        row=row_for('confirmation',seed,arm)
        if row is None or row.get('return_code')!=0 or row.get('run_status')!='scored' or row.get('grid_id')!='heat-confirmatory-512x256-v1': return False
        if not finite_flat_metrics(row.get('metrics_path','')): return False
        try: report=json.loads(pathlib.Path(row.get('cost_path','')).read_text())
        except Exception: return False
        if report.get('arm')!=arm or report.get('seed')!=seed or report.get('grid_id')!='heat-confirmatory-512x256-v1': return False
        metrics=report.get('metrics')
        if not isinstance(metrics,dict) or not metrics or not all(isinstance(v,(int,float)) and not isinstance(v,bool) and np.isfinite(v) for v in metrics.values()): return False
    return True
campaign_complete=smoke_campaign_valid() and tier2_campaign_valid() and confirmation_campaign_valid() and DEVICE_SPENT<=DEVICE_CAP
save_ledger('complete' if campaign_complete else 'incomplete_validation')
print(json.dumps({'status':ledger['status'],'quota_hours':ledger.get('kaggle_quota_hours_estimate'),'aggregate_gpu_hours':DEVICE_SPENT/3600,'records':len(ledger['runs'])},indent=2))


## Comparison caveats

The 512×256 confirmation score is computed on CUDA in a separate scorer only after every training run finishes. Its grid is generated there and is never read by training or used for selection. H-FNN is a plain PyTorch port of the tutorial's network/optimizer recipe; it is neither DeepXDE runtime execution nor a reproduction. Its soft IC/BC losses are replaced by the exact hard trace used by H-RBF/H-BS; alpha, pools, precision, loss scaling, development grid, and implementation are adapted to the AF contract. RBF and B-spline also run from a static source snapshot in a new full-batch loop. Historical cohorts remain contextual; no automatic SOTA claim is made.